# Gộp và làm sạch dữ liệu GBIF 2020–2026

Đặt 7 file `gbif_vn_plants_<năm>.csv` (tải từ các notebook cào) vào thư mục `DATA_DIR`.

In [ ]:
DATA_DIR = "data"

In [ ]:
import glob
import pandas as pd

files = sorted(glob.glob(f"{DATA_DIR}/gbif_vn_plants_*.csv"))
print(len(files), "file:", [f.split("_")[-1][:4] for f in files])
df = pd.concat([pd.read_csv(f, low_memory=False) for f in files], ignore_index=True)
print("Truoc khi loai trung:", len(df))
df = df.drop_duplicates("gbifID")
print("Sau khi loai trung:", len(df))

## Tạo cột phụ

In [ ]:
THREATENED = {"CR", "EN", "VU"}

df["is_assessed"] = df["iucnRedListCategory"].notna()
df["is_threatened"] = df["iucnRedListCategory"].isin(THREATENED)
df["source_type"] = df["basisOfRecord"].map({
    "PRESERVED_SPECIMEN": "Tieu ban",
    "HUMAN_OBSERVATION": "Quan sat",
    "OCCURRENCE": "Quan sat",
    "MACHINE_OBSERVATION": "Quan sat",
    "LIVING_SPECIMEN": "Mau song",
}).fillna("Khac")
df["year_month"] = pd.to_datetime(
    dict(year=df["year"], month=df["month"].fillna(1).astype(int), day=1), errors="coerce"
)

print(df[["is_assessed", "is_threatened"]].mean().round(3).to_string())
print()
print(df["source_type"].value_counts().to_string())
print()
print("Ty le thieu theo cot:")
print(df.isna().mean().round(2).sort_values(ascending=False).to_string())

## Lưu file dùng cho dashboard

In [ ]:
df.to_csv("vn_plants_gbif_2020_2026.csv", index=False, encoding="utf-8-sig")
print("Da luu vn_plants_gbif_2020_2026.csv:", df.shape)